# Простая предобработка и построение бейзлайна. Эксперименты по предобработке

In [66]:
import numpy as np
import pandas as pd

In [67]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

In [68]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [69]:
train_data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [70]:
test_data.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

Сделаем простую предобработку данных  
1) дропнем PassengerId, Name, Ticket, Cabin
2) заполним пропуски в embarked модой S
3) заполним пропуски в age медианой
4) sex переведем из male/female к 0/1
5) embarked превратим в 0, 1, 2

Дропнем PassengerId, Name, Ticket, Cabin

In [71]:
train_data = train_data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
train_data.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

Заполним пропуски в Embarked модой (S)

In [72]:
mode_embarked = train_data['Embarked'].mode()[0]
train_data['Embarked'] = train_data['Embarked'].fillna(mode_embarked)

train_data['Embarked'].isna().sum()

np.int64(0)

Заполним пропуски в Age медианой

In [73]:
median_age = train_data['Age'].median()
train_data['Age'] = train_data['Age'].fillna(median_age)

In [74]:
train_data.isna().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [75]:
train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


sex переведем из male/female к 0/1

In [76]:
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1})

embarked превратим в 0, 1, 2

In [77]:
train_data['Embarked'] = train_data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

In [78]:
train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0


Все колонки числовые (one-hot, разбитие на бины и масштабирование специально не применяется, чтобы посмотреть, как эти устоявшиеся методики будут влиять на скор), пропусков нет, нужно сделать с test то же самое, а потом можно переходить к построению модели

In [79]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [80]:
test_data.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [81]:
test_data = test_data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
test_data['Embarked'] = test_data['Embarked'].fillna(mode_embarked)
test_data['Age'] = test_data['Age'].fillna(median_age)
test_data['Fare'] = test_data['Fare'].fillna(train_data['Fare'].median())
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})
test_data['Embarked'] = test_data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

In [82]:
test_data.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,0,34.5,0,0,7.8292,2
1,3,1,47.0,1,0,7.0000,0
2,2,0,62.0,0,0,9.6875,2
3,3,0,27.0,0,0,8.6625,0
4,3,1,22.0,1,1,12.2875,0


In [83]:
train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0


Создание модели логистической регрессии

In [84]:
import sklearn
from sklearn.model_selection import train_test_split

In [85]:
X = train_data.drop(['Survived'], axis=1)
y = train_data['Survived']

X, y

(     Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
 0         3    0  22.0      1      0   7.2500         0
 1         1    1  38.0      1      0  71.2833         1
 2         3    1  26.0      0      0   7.9250         0
 3         1    1  35.0      1      0  53.1000         0
 4         3    0  35.0      0      0   8.0500         0
 ..      ...  ...   ...    ...    ...      ...       ...
 886       2    0  27.0      0      0  13.0000         0
 887       1    1  19.0      0      0  30.0000         0
 888       3    1  28.0      1      2  23.4500         0
 889       1    0  26.0      0      0  30.0000         1
 890       3    0  32.0      0      0   7.7500         2
 
 [891 rows x 7 columns],
 0      0
 1      1
 2      1
 3      1
 4      0
       ..
 886    0
 887    1
 888    0
 889    1
 890    0
 Name: Survived, Length: 891, dtype: int64)

In [86]:
test_size = 0.2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
len(X_train), len(X_test), len(y_train), len(y_test)

(712, 179, 712, 179)

In [87]:
from sklearn.linear_model import LogisticRegression

In [88]:
basic_logistic_regression = LogisticRegression()

In [89]:
basic_logistic_regression.fit(X_train, y_train)

c:\Users\rar22\ML\kaggle\titanic\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [90]:
from sklearn.metrics import accuracy_score

y_pred = basic_logistic_regression.predict(X_test)
baseline_score = accuracy_score(y_test, y_pred)
print(f"Базовый скор: {baseline_score}")

Базовый скор: 0.7988826815642458


Применение обученной модели на тестовом датасете и создание сабмита для кагла

In [91]:
test_passenger_ids = pd.read_csv("data/test.csv")['PassengerId']

predictions = basic_logistic_regression.predict(test_data)

submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': predictions
})
submission.to_csv('submission.csv', index=False)


In [92]:
submission

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [93]:
submission.shape

(418, 2)

Скор после загрузки на кагл  
Score: 0.76315 (split: 0.8/0.2)  
  
Когда до этого был split 0.67/0.33, был вот такой скор:  
0.76794

Создадим функции базовой предобработки, обучения и инференса модели

In [94]:
def preprocess_data(df, is_train, mode_embarked=None, median_age=None, median_fare=None):
    df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

    if is_train:
        mode_embarked = df['Embarked'].mode()[0]
        median_age = df['Age'].median()
        median_fare = df['Fare'].median()

    df['Embarked'] = df['Embarked'].fillna(mode_embarked)
    df['Age'] = df['Age'].fillna(median_age)
    df['Fare'] = df['Fare'].fillna(median_fare)
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

    if is_train:
        return df, mode_embarked, median_age, median_fare
    return df

In [95]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def train_and_evaluate(X, y, model, random_state=42, test_size=test_size):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return model, accuracy_score(y_test, y_pred)

Функция сравнения скора на бейзлайне и текущего скора

In [96]:
def score_comparison(baseline_score, new_score):
    print(f"Базовый скор: {baseline_score}")
    print(f"Новый скор: {new_score}")
    if new_score > baseline_score:
        print(f"Новый скор больше базового")
        print(f"Прирост составил: {(new_score - baseline_score):.6f}")
    elif new_score < baseline_score:
        print(f"Новый скор меньше базового")
        print(f"Ухудшение составило: {(baseline_score - new_score):.6f}")
    else:
        print(f"Улучшений нет")

Проверка функций

In [97]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

train_data, mode_embarked, median_age, median_fare = preprocess_data(train_data, is_train=True)
test_data = preprocess_data(test_data, is_train=False, mode_embarked=mode_embarked, median_age=median_age, median_fare=median_fare)


In [98]:
X = train_data.drop(['Survived'], axis=1)
y = train_data['Survived']

model, score = train_and_evaluate(X, y, LogisticRegression())
print(f"Скор на трейне: {score}")


Скор на трейне: 0.7988826815642458


c:\Users\rar22\ML\kaggle\titanic\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Применим нормализацию для Age, Fare, SibSp, Parch

In [99]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

train_data, mode_embarked, median_age, median_fare = preprocess_data(train_data, is_train=True)
test_data = preprocess_data(test_data, is_train=False, mode_embarked=mode_embarked, median_age=median_age, median_fare=median_fare)

In [100]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['Age', 'Fare', 'SibSp', 'Parch']

scaler = StandardScaler()
train_data[scale_cols] = scaler.fit_transform(train_data[scale_cols])
test_data[scale_cols] = scaler.transform(test_data[scale_cols])

train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,-0.565736,0.432793,-0.473674,-0.502445,0
1,1,1,1,0.663861,0.432793,-0.473674,0.786845,1
2,1,3,1,-0.258337,-0.474545,-0.473674,-0.488854,0
3,1,1,1,0.433312,0.432793,-0.473674,0.420730,0
4,0,3,0,0.433312,-0.474545,-0.473674,-0.486337,0


In [101]:
X = train_data.drop(['Survived'], axis=1)
y = train_data['Survived']

model, score = train_and_evaluate(X, y, LogisticRegression())
score_comparison(baseline_score, score)

Базовый скор: 0.7988826815642458
Новый скор: 0.7988826815642458
Улучшений нет


Применили только нормализацию к числовым колонкам, но улучшений не произошло

Выполним one-hot encoding категориальных фичей

In [102]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

train_data, mode_embarked, median_age, median_fare = preprocess_data(train_data, is_train=True)
test_data = preprocess_data(test_data, is_train=False, mode_embarked=mode_embarked, median_age=median_age, median_fare=median_fare)

In [103]:
categorical_cols = ['Embarked', 'Pclass']

train_data = pd.get_dummies(train_data, columns=categorical_cols)
test_data = pd.get_dummies(test_data, columns=categorical_cols)

test_data = test_data.reindex(columns=train_data.drop('Survived', axis=1).columns, fill_value=0)

In [104]:
test_data.head()

,Sex,Age,SibSp,Parch,Fare,Embarked_0,Embarked_1,Embarked_2,Pclass_1,Pclass_2,Pclass_3
0,0,34.5,0,0,7.8292,False,False,True,False,False,True
1,1,47.0,1,0,7.0000,True,False,False,False,False,True
2,0,62.0,0,0,9.6875,False,False,True,False,True,False
3,0,27.0,0,0,8.6625,True,False,False,False,False,True
4,1,22.0,1,1,12.2875,True,False,False,False,False,True


In [105]:
X = train_data.drop(['Survived'], axis=1)
y = train_data['Survived']

model, score = train_and_evaluate(X, y, LogisticRegression())
score_comparison(baseline_score, score)

Базовый скор: 0.7988826815642458
Новый скор: 0.7988826815642458
Улучшений нет


Выполнили one-hot encoding категориальных фичей, улучшений не произошло

Попробуем выполнить максимальную предобработку с генерацией новых фичей, нормированием данных и one-hot кодированием

Сгенерируем новые фичи из Name, Ticket и Cabin (как в eda_and_hypotheses): Title (титул, сгруппированный до Mr/Mrs/Miss/Master/Rare), TicketGroupSize (размер группы по билету) и HasCabin (наличие каюты)

Не берём FarePerPerson и TicketPrefix: по матрице корреляций из eda_and_hypotheses FarePerPerson сильно коррелирует с Fare (0.84) и Pclass (0.66), а TicketPrefix там же отмечен как более слабый и менее интерпретируемый сигнал

Пропуски в Age заполним медианой по Title, а не общей медианой - Title несёт сигнал о поле и возрасте (эта идея тоже из eda_and_hypotheses)

In [106]:
def extract_title(names):
    title = names.str.extract(r' ([A-Za-z]+)\.')[0]
    title = title.replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    common_titles = {'Mr', 'Mrs', 'Miss', 'Master'}
    return title.where(title.isin(common_titles), 'Rare')


def engineer_features(df, ticket_counts):
    df = df.copy()
    df['Title'] = extract_title(df['Name'])
    df['TicketGroupSize'] = df['Ticket'].map(ticket_counts).fillna(1)
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    return df.drop(['Name', 'Ticket', 'Cabin'], axis=1)

In [107]:
def preprocess_data_advanced(df, is_train, artifacts=None):
    df = df.drop(['PassengerId'], axis=1)

    if is_train:
        artifacts = {'ticket_counts': df['Ticket'].value_counts()}

    df = engineer_features(df, artifacts['ticket_counts'])

    if is_train:
        artifacts['mode_embarked'] = df['Embarked'].mode()[0]
        artifacts['median_age_by_title'] = df.groupby('Title')['Age'].median()
        artifacts['median_fare'] = df['Fare'].median()

    df['Embarked'] = df['Embarked'].fillna(artifacts['mode_embarked'])
    df['Fare'] = df['Fare'].fillna(artifacts['median_fare'])
    df['Age'] = df['Age'].fillna(df['Title'].map(artifacts['median_age_by_title']))
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

    return (df, artifacts) if is_train else df

Применим предобработку, one-hot кодирование категориальных фичей (Pclass, Embarked, Title) и нормирование числовых (Age, Fare, SibSp, Parch, TicketGroupSize)

In [108]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

train_data, artifacts = preprocess_data_advanced(train_data, is_train=True)
test_data = preprocess_data_advanced(test_data, is_train=False, artifacts=artifacts)

categorical_cols = ['Pclass', 'Embarked', 'Title']
train_data = pd.get_dummies(train_data, columns=categorical_cols)
test_data = pd.get_dummies(test_data, columns=categorical_cols)
test_data = test_data.reindex(columns=train_data.drop('Survived', axis=1).columns, fill_value=0)

from sklearn.preprocessing import StandardScaler

scale_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'TicketGroupSize']

scaler = StandardScaler()
train_data[scale_cols] = scaler.fit_transform(train_data[scale_cols])
test_data[scale_cols] = scaler.transform(test_data[scale_cols])

train_data.head()

,Survived,Sex,Age,SibSp,Parch,Fare,TicketGroupSize,HasCabin,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,0,-0.557460,0.432793,-0.473674,-0.502445,-0.579162,0,False,False,True,False,False,True,False,False,True,False,False
1,1,1,0.649091,0.432793,-0.473674,0.786845,-0.579162,1,True,False,False,True,False,False,False,False,False,True,False
2,1,1,-0.255822,-0.474545,-0.473674,-0.488854,-0.579162,0,False,False,True,False,False,True,False,True,False,False,False
3,1,1,0.422862,0.432793,-0.473674,0.420730,0.155928,1,True,False,False,False,False,True,False,False,False,True,False
4,0,0,0.422862,-0.474545,-0.473674,-0.486337,-0.579162,0,False,False,True,False,False,True,False,False,True,False,False


In [109]:
X = train_data.drop(['Survived'], axis=1)
y = train_data['Survived']

model, score = train_and_evaluate(X, y, LogisticRegression())
score_comparison(baseline_score, score)

Базовый скор: 0.7988826815642458
Новый скор: 0.8268156424581006
Новый скор больше базового
Прирост составил: 0.027933


Полная предобработка данных дает прирост скора 0.027933 для логистической регрессии с дефолтными гиперпараметрами

Зафиксируем финальный препроцессинг (тот, что дал лучший результат в экспериментах выше) и соберём финальные X_train/y_train (для обучения и валидации) и X_test (реальный тестовый сет для сабмита)

In [110]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

train_data, artifacts = preprocess_data_advanced(train_data, is_train=True)
test_data = preprocess_data_advanced(test_data, is_train=False, artifacts=artifacts)

categorical_cols = ['Pclass', 'Embarked', 'Title']
train_data = pd.get_dummies(train_data, columns=categorical_cols)
test_data = pd.get_dummies(test_data, columns=categorical_cols)
test_data = test_data.reindex(columns=train_data.drop('Survived', axis=1).columns, fill_value=0)

scale_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'TicketGroupSize']

scaler = StandardScaler()
train_data[scale_cols] = scaler.fit_transform(train_data[scale_cols])
test_data[scale_cols] = scaler.transform(test_data[scale_cols])

X_train = train_data.drop(['Survived'], axis=1)
y_train = train_data['Survived']
X_test = test_data

X_train.shape, X_test.shape

((891, 18), (418, 18))

Соберём Stratified K-Fold харнесс: функция принимает модель, X, y и возвращает accuracy по каждому фолду (StratifiedKFold сохраняет соотношение классов в каждом фолде, в отличие от обычного KFold)

In [111]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

def cross_validate_model(model, X, y, n_splits=5, random_state=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    return cross_val_score(model, X, y, cv=cv, scoring='accuracy')

Прогоним LogisticRegression через харнесс и сравним скор на 1 фолде со средним по 5 фолдам

In [112]:
cv_scores = cross_validate_model(LogisticRegression(), X_train, y_train)

print(f"Скор по фолдам: {cv_scores}")
print(f"Скор на 1 фолде: {cv_scores[0]:.5f}")
print(f"Средний скор по 5 фолдам: {cv_scores.mean():.5f} +- {cv_scores.std():.5f}")

Скор по фолдам: [0.8603352  0.81460674 0.8258427  0.8258427  0.84269663]
Скор на 1 фолде: 0.86034
Средний скор по 5 фолдам: 0.83386 +- 0.01599


Скор на одном фолде 0.86 заметно оптимистичнее среднего по 5 фолдам 0.83, оценивать модель по 5 фолдам надежнее

Закроем пункт чеклиста "сравните скор на лидерборде у 1 и 5 фолдов". Модель и препроцессинг уже зафиксированы выше (X_train, y_train, X_test, LogisticRegression) — дальше только различается способ валидации/обучения перед сабмитом

Сабмит "1 фолд": обучаем модель на одном holdout-сплите и предсказываем на test.csv

In [113]:
X_train_1fold, X_val_1fold, y_train_1fold, y_val_1fold = train_test_split(X_train, y_train, test_size=test_size, random_state=42)

model_1fold = LogisticRegression()
model_1fold.fit(X_train_1fold, y_train_1fold)

val_score_1fold = accuracy_score(y_val_1fold, model_1fold.predict(X_val_1fold))
print(f"Скор на валидационном сплите (1 фолд): {val_score_1fold:.5f}")

test_passenger_ids = pd.read_csv("data/test.csv")['PassengerId']
predictions_1fold = model_1fold.predict(X_test)

submission_1fold = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': predictions_1fold
})
submission_1fold.to_csv('submission_1fold.csv', index=False)

Скор на валидационном сплите (1 фолд): 0.82682


Паблик скор на кагле для submission_1fold.csv: 0.77033

Сабмит "5 фолдов": обучаем 5 моделей по StratifiedKFold (каждая на своей train-части) и усредняем предсказанные вероятности на test.csv

In [114]:
from sklearn.base import clone

def train_kfold_and_predict(model, X, y, X_test, n_splits=5, random_state=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    test_proba = np.zeros(len(X_test))

    for train_idx, _ in cv.split(X, y):
        # clone делает чистую копию модели с теми же гиперпараметрами, но без обучения.
        # Без этого все 5 фолдов дообучали бы один и тот же объект model,
        # и для некоторых моделей (например, с warm_start=True) это испортило бы независимость фолдов.
        fold_model = clone(model)
        fold_model.fit(X.iloc[train_idx], y.iloc[train_idx])
        test_proba += fold_model.predict_proba(X_test)[:, 1] / n_splits

    return (test_proba >= 0.5).astype(int)

In [115]:
predictions_5fold = train_kfold_and_predict(LogisticRegression(), X_train, y_train, X_test)

submission_5fold = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': predictions_5fold
})
submission_5fold.to_csv('submission_5fold.csv', index=False)

Паблик скор на кагле для submission_5fold.csv: 0.77033 — совпадает с 1-фолдом. Локальный CV-скор при этом был выше (0.834 ± 0.016)

Проверим, отличаются ли предсказания 1-фолда и 5-фолдов, или скор совпал случайно

In [116]:
diff_mask = submission_1fold['Survived'] != submission_5fold['Survived']
print(f"Отличаются предсказания в {diff_mask.sum()} строках из {len(diff_mask)}")

Отличаются предсказания в 6 строках из 418


Предсказания отличаются в 6 строках, но скор совпал случайно. На 418 строках LB не может надёжно различить 1 fold и 5 folds, буду ориентироваться на локальный CV-скор, а не гнаться за LB на каждом эксперименте